# Neural Networks and Deep Learning — Seminar 14: Multimodal and vision-language models; synthesis of the course
Prof. dr. Daniel Traian Pele — Bucharest University of Economic Studies (ASE); Romanian Academy, Institute for Economic Forecasting (IPE)

[Open in Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L14_seminar.ipynb)

Seminar tasks on real data from `../data/clean` (see `../data/SOURCES.md`). Write your code in the empty cells under each task; keep the seed fixed so that your numbers can be compared.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Setup

In [ ]:
RES = '../lectures/L14/results/'
import torch, torchvision
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
root = '../data/raw/vision'
os.makedirs(root, exist_ok=True)
ES = torchvision.datasets.EuroSAT(root, download=True)
Z = pd.read_csv(RES + 'eurosat_zeroshot.csv')
CLS = ES.classes
print(len(ES), 'images;', len(Z), 'test images in the lecture split')

In [ ]:
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
def img_emb(idx):
    out = []
    with torch.no_grad():
        for j in range(0, len(idx), 64):
            px = proc(images=[ES[int(i)][0] for i in idx[j:j + 64]], return_tensors='pt')['pixel_values'].to(DEVICE)
            e = clip.get_image_features(pixel_values=px)
            e = e if isinstance(e, torch.Tensor) else e.pooler_output
            out.append(torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)
def txt_emb(texts):
    with torch.no_grad():
        t = proc(text=texts, return_tensors='pt', padding=True).to(DEVICE)
        e = clip.get_text_features(**t)
        e = e if isinstance(e, torch.Tensor) else e.pooler_output
        return torch.nn.functional.normalize(e.float(), dim=-1).cpu().numpy()
sample = Z.sample(500, random_state=SEED)
E = img_emb(sample['index'].values)
y = sample['true'].map({c: i for i, c in enumerate(CLS)}).values
NAMES = {'AnnualCrop': 'annual crop land', 'Forest': 'forest', 'HerbaceousVegetation': 'brushland or shrubland', 'Highway': 'highway or road',
         'Industrial': 'industrial buildings or commercial buildings', 'Pasture': 'pasture land', 'PermanentCrop': 'permanent crop land',
         'Residential': 'residential buildings or homes or apartments', 'River': 'river', 'SeaLake': 'lake or sea'}

**Task 1.** Build a zero-shot classifier with your own template (for example 'an aerial image of {}') and report its accuracy on the 500 sampled images.

In [ ]:
# your code here

**Task 2.** Average the embeddings of three templates per class (prompt ensemble). Does the accuracy improve over the single template?

In [ ]:
# your code here

**Task 3.** Train a logistic regression on CLIP ViT-B/32 embeddings with 4 and with 32 images per class (seed fixed). Plot the two accuracies next to the zero-shot one.

In [ ]:
# your code here

**Task 4.** From `eurosat_zeroshot.csv`, list the three most frequent confusions of the best zero-shot model.

In [ ]:
# your code here

**Task 5.** For the chart-regime task, compute a 95 % bootstrap interval of the difference in AUC between the numeric model and the CLIP probe on the test windows.

In [ ]:
# your code here